In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/transactions.csv", parse_dates=["timestamp"])
df.head()

,transaction_id,user_id,timestamp,txn_type,amount,merchant_id,merchant_category,city,latitude,longitude,device_id,is_fraud,fraud_pattern
0,T000001,U0015,2026-01-01 03:30:05.314749442,remittance,67575.25,M038,education,Bhaktapur,27.6663,85.4270,D85ad8844,1,sim_swap
1,T000002,U0200,2026-01-01 05:04:30.300764748,POS,1890.97,M004,food,Kathmandu,27.7072,85.3136,Db11ce396,0,none
2,T000003,U0083,2026-01-01 07:30:55.647457094,POS,1375.14,M069,transport,Kathmandu,27.6954,85.2984,D06bebd6b,0,none
3,T000004,U0190,2026-01-01 09:16:25.053401383,POS,3041.48,M067,utility,Kathmandu,27.6988,85.3242,D673cf002,0,none
4,T000005,U0122,2026-01-01 09:21:19.306733881,QR,1033.52,M023,utility,Nepalgunj,28.0493,81.6297,D4bda4ae6,0,none


In [2]:
df.groupby("txn_type")["is_fraud"].mean().round(3)

txn_type
POS           0.043
QR            0.058
remittance    0.065
Name: is_fraud, dtype: float64

## Finding 1: fraud rate by transaction type

- POS is 4.3%, QR is 5.8%, remittance is 6.5%.
- The rates are close. Remittance is slightly higher.
- Caveat: this comes from how the synthetic generator was written. It is not a finding about real fraud in Nepal.

In [3]:
df.groupby("fraud_pattern")["amount"].median().round(0).sort_values()

fraud_pattern
velocity_burst        659.0
new_merchant          660.0
none                 1263.0
cold_start           4139.0
amount_spike        13155.0
location_anomaly    15418.0
sim_swap            62834.0
Name: amount, dtype: float64

## Finding 2: median amount by fraud pattern

- sim_swap has a median of 62,834, about 50 times the legitimate median of 1,263.
- location_anomaly and amount_spike are about 10 to 12 times higher.
- velocity_burst and new_merchant look low only because they use POS and QR, never remittance.
- Caveat: this table mixes fraud pattern with transaction type. The next step compares within each type.

In [4]:
df.pivot_table(
    index="fraud_pattern",
    columns="txn_type",
    values="amount",
    aggfunc="median",
).round(0)

txn_type,POS,QR,remittance
fraud_pattern,,,
amount_spike,16817.0,4511.0,130895.0
cold_start,4629.0,1709.0,49629.0
location_anomaly,1419.0,425.0,21168.0
new_merchant,1770.0,474.0,NaN
none,1531.0,414.0,20427.0
sim_swap,8396.0,2343.0,70108.0
velocity_burst,1373.0,469.0,NaN


## Finding 3: median amount by pattern, split by transaction type

- Within the same type, amount_spike is about 11x legitimate for POS and QR, and 6x for remittance.
- sim_swap is 3x to 6x higher and cold_start is 2x to 4x higher.
- location_anomaly, new_merchant and velocity_burst are within about 15% of legitimate amounts.
- Conclusion: amount alone cannot catch these three. They need distance, merchant rarity, and transaction-count features.
- Lesson: the first table (Finding 2) mixed pattern with transaction type. Always compare within type.

In [5]:
df["merchant_txn_count"] = df.groupby("merchant_id")["transaction_id"].transform("count")
df.groupby("fraud_pattern")["merchant_txn_count"].median()

fraud_pattern
amount_spike        119.0
cold_start           10.0
location_anomaly    203.0
new_merchant         10.0
none                121.0
sim_swap             10.0
velocity_burst      143.0
Name: merchant_txn_count, dtype: float64

## Finding 4: merchant transaction count by fraud pattern

- cold_start, new_merchant and sim_swap use merchants with a median of 10 transactions. Legitimate transactions use merchants with a median of 121.
- amount_spike, velocity_burst and location_anomaly use ordinary merchants, so merchant rarity cannot catch them.
- This matches the Graph-inspired Risk Agent idea: rare merchants carry risk.
- Caveat: the generator was written to produce this pattern. Real fraud will be less clean.
- Caveat: the count uses the whole dataset, including future rows. A real model must use only past transactions.

In [6]:
df["hour"] = df["timestamp"].dt.hour
df["night"] = df["hour"].between(0, 5)
df.groupby("fraud_pattern")["night"].mean().round(2)

fraud_pattern
amount_spike        0.18
cold_start          0.18
location_anomaly    0.01
new_merchant        0.00
none                0.01
sim_swap            1.00
velocity_burst      0.00
Name: night, dtype: float64

## Finding 5: share of transactions between 0 and 5 AM, by fraud pattern

- sim_swap happens 100% at night. Legitimate transactions are 1%.
- sim_swap is now flagged by three signals: amount, merchant rarity, and time of day.
- amount_spike (18%) and cold_start (18%) are partly at night.
- Caveat: the cold_start night share is a side effect of how the generator places timestamps. It is not an intended signal.
- location_anomaly, velocity_burst and new_merchant show no night signal.

In [7]:
home = df.groupby("user_id")["city"].agg(lambda s: s.mode()[0])
df["home_city"] = df["user_id"].map(home)
df["away"] = df["city"] != df["home_city"]
df.groupby("fraud_pattern")["away"].mean().round(2)

fraud_pattern
amount_spike        0.00
cold_start          0.00
location_anomaly    1.00
new_merchant        0.00
none                0.04
sim_swap            0.43
velocity_burst      0.00
Name: away, dtype: float64

## Finding 6: share of transactions away from the user's home city, by fraud pattern

- location_anomaly is away from home 100% of the time. Legitimate transactions are away 4%.
- sim_swap is away 43% of the time, so location is only a partial signal for it.
- The 4% of honest travel is a source of false positives for any location rule.
- Method: home city is each user's most common city, calculated without using fraud labels.
- Caveat: this uses all rows to find home city. A real model must use only past transactions.
- Caveat: a better feature is distance from home, plus time since the last transaction ("impossible travel").

In [8]:
df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)
df["gap_sec"] = df.groupby("user_id")["timestamp"].diff().dt.total_seconds()
df["rapid"] = df["gap_sec"] < 60
df.groupby("fraud_pattern")["rapid"].mean().round(2)

fraud_pattern
amount_spike        0.00
cold_start          0.00
location_anomaly    0.00
new_merchant        0.00
none                0.00
sim_swap            0.00
velocity_burst      0.84
Name: rapid, dtype: float64

## Finding 7: share of transactions within 60 seconds of the user's previous one, by fraud pattern

- velocity_burst is 84%. Every other pattern, including legitimate transactions, is 0%.
- The first transaction of each burst cannot be flagged this way, which explains why it is 84% and not 100%.
- Method: the gap looks backward only, so it uses past data only.
- Caveat: legitimate users never trigger this rule in the synthetic data. Real users sometimes do, so real false positives would be higher.

## Summary

- Each of the six fraud patterns can be detected by at least one simple signal: amount, merchant rarity, time of night, distance from home, or transaction speed.
- No single signal catches all six, which supports using several agents together.
- These results describe the synthetic generator I wrote. They do not describe real fraud in Nepal.
- Next step: build these signals as proper features, using only past data, and test them with a baseline model.

In [9]:
order = ["none", "amount_spike", "sim_swap", "cold_start",
         "new_merchant", "location_anomaly", "velocity_burst"]

summary = df.groupby("fraud_pattern").agg(
    rows=("transaction_id", "count"),
    median_amount=("amount", "median"),
    median_merchant_count=("merchant_txn_count", "median"),
    night_share=("night", "mean"),
    away_share=("away", "mean"),
    rapid_share=("rapid", "mean"),
).round(2).reindex(order)

summary

,rows,median_amount,median_merchant_count,night_share,away_share,rapid_share
fraud_pattern,,,,,,
none,8050,1262.52,121.0,0.01,0.04,0.00
amount_spike,45,13154.98,119.0,0.18,0.00,0.00
sim_swap,67,62833.84,10.0,1.00,0.43,0.00
cold_start,45,4138.77,10.0,0.18,0.00,0.00
new_merchant,90,659.71,10.0,0.00,0.00,0.00
location_anomaly,90,15417.52,203.0,0.01,1.00,0.00
velocity_burst,113,658.88,143.0,0.00,0.00,0.84


## Finding 8: all signals in one table

- Each fraud pattern stands out on at least one column compared with the `none` row.
- median_amount mixes transaction types, so it is misleading on its own (see Finding 3).
- rapid_share is 0 for every row except velocity_burst, which is why speed is such a clean signal here.
- Caveat: all of this describes the synthetic generator, not real fraud.

## Which signal catches which pattern

| Pattern | Amount | Merchant rarity | Night | Away from home | Speed |
|---|---|---|---|---|---|
| amount_spike | Strong | - | - | - | - |
| sim_swap | Strong | Strong | Strong | Partial | - |
| cold_start | Partial | Strong | - | - | - |
| new_merchant | - | Strong | - | - | - |
| location_anomaly | - | - | - | Strong | - |
| velocity_burst | - | - | - | - | Strong |

Reading it: no column is Strong for every row, so no single signal is enough. Several signals together are needed. This supports the multi-agent design in the paper.